# Adult dataset operating modes

This notebook demonstrates MIMIC's main operating modes on the Adult/Census Income dataset.

Set `N_ROWS` to control the working sample size. The default is intentionally small enough for quick experimentation.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, mean_absolute_error

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import MIMIC, GenerationPolicy, RandomForestPathEncoder, MixedFeatureDecoder 
from mimic import ForestConditionalSampler
from mimic.diagnostics import binary_classification_diagnostics, classical_mds_2d, pairwise_feature_plot

RANDOM_STATE = 42
N_ROWS = 1000
rng = np.random.default_rng(RANDOM_STATE)

## Load and clean Adult

Adult has no real ID column, but it does contain sampling-weight and education-code columns that are not useful for this demonstration. `fnlwgt` has a very wide range and `education-num` duplicates information already present in `education`, so both are excluded.

In [2]:
adult = fetch_openml("adult", version=2, as_frame=True)
raw = adult.frame.copy()
raw = raw.replace("?", np.nan)

drop_columns = ["fnlwgt", "education-num"]
clean = raw.drop(columns=drop_columns).dropna().reset_index(drop=True)
clean["income"] = clean["class"].astype(str)
clean = clean.drop(columns=["class"])

sampled = clean.sample(n=2 * N_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
df = sampled.iloc[:N_ROWS].reset_index(drop=True)
test_df = sampled.iloc[N_ROWS:].reset_index(drop=True)

regression_columns = ["age", "hours-per-week", "capital-gain", "capital-loss"]
classification_columns = [c for c in df.columns if c not in regression_columns]

dataset_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(df), len(test_df)],
        "columns": [df.shape[1], test_df.shape[1]],
        "income_<=50K": [
            df["income"].value_counts(normalize=True).get("<=50K", 0.0),
            test_df["income"].value_counts(normalize=True).get("<=50K", 0.0),
        ],
        "income_>50K": [
            df["income"].value_counts(normalize=True).get(">50K", 0.0),
            test_df["income"].value_counts(normalize=True).get(">50K", 0.0),
        ],
    }
)

display(dataset_summary.style.format({"income_<=50K": "{:.1%}", "income_>50K": "{:.1%}"}))
display(df.head())

,split,rows,columns,income_<=50K,income_>50K
0,train,1000,13,74.4%,25.6%
1,test,1000,13,74.8%,25.2%


,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,19,Self-emp-not-inc,10th,Married-spouse-absent,Adm-clerical,Unmarried,Amer-Indian-Eskimo,Female,0,0,40,United-States,<=50K
1,45,Private,HS-grad,Married-civ-spouse,Farming-fishing,Husband,Amer-Indian-Eskimo,Male,0,0,40,United-States,<=50K
2,47,Private,Assoc-voc,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K
3,23,Private,Some-college,Never-married,Craft-repair,Other-relative,White,Male,0,0,40,United-States,<=50K
4,53,Local-gov,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K


## Fit from clean data

The model is fitted on clean complete rows. Later cells create perturbed copies to test each operating mode.

In [3]:
mimic = MIMIC(
    regression_columns=regression_columns,
    classification_columns=classification_columns,
    encoder=RandomForestPathEncoder(n_estimators=50, embedding_dim=100, random_state=RANDOM_STATE),
    decoder=ForestConditionalSampler(n_estimators=120, random_state=2),
    policy=GenerationPolicy(method="displacement", neighbour_mode="mutual", n_neighbors=5, lambda_range=(0.25, 0.75)),
    n_bootstrap=2,
    random_state=RANDOM_STATE,
)
mimic.fit(df)
embedding = mimic.transform(df)
fit_summary = pd.DataFrame(
    {
        "quantity": ["training_rows", "embedding_rows", "embedding_dimensions", "modelled_columns"],
        "value": [len(df), embedding.shape[0], embedding.shape[1], len(regression_columns) + len(classification_columns)],
    }
)
display(fit_summary)

KeyboardInterrupt: 

## 1. Missing-value imputation

Hide known values and measure how well MIMIC reconstructs them.

In [ ]:
impute_columns = ["age", "hours-per-week", "workclass", "marital-status"]
masked = test_df.copy()
truth = []
for col in impute_columns:
    rows = rng.choice(masked.index, size=max(20, N_ROWS // 20), replace=False)
    truth.append(pd.DataFrame({"row": rows, "column": col, "true": test_df.loc[rows, col].to_numpy()}))
    masked.loc[rows, col] = np.nan
truth = pd.concat(truth, ignore_index=True)

reconstructed = mimic.impute(masked, columns=impute_columns)
truth["predicted"] = [reconstructed.loc[r, c] for r, c in zip(truth["row"], truth["column"])]

imputation_scores = []
for col in impute_columns:
    part = truth[truth["column"] == col]
    if col in regression_columns:
        score = mean_absolute_error(part["true"].astype(float), part["predicted"].astype(float))
        metric = "MAE"
    else:
        score = accuracy_score(part["true"].astype(str), part["predicted"].astype(str))
        metric = "accuracy"
    imputation_scores.append({"column": col, "metric": metric, "score": score})

imputation_scores = pd.DataFrame(imputation_scores)
imputation_examples = truth.head(12)
display(imputation_scores.style.format({"score": "{:.3f}"}))
display(imputation_examples)

## 2. Data inconsistency check

Inject unrealistic numerical values beyond two standard deviations from the clean distribution and rank entries by prediction discrepancy.

In [ ]:
corrupted = test_df.copy()
for col in ["age", "hours-per-week"]:
    corrupted[col] = corrupted[col].astype(float)

corruption_records = []
for col in ["age", "hours-per-week"]:
    rows = rng.choice(corrupted.index, size=max(10, N_ROWS // 50), replace=False)
    mu = test_df[col].mean()
    sigma = test_df[col].std()
    corrupted.loc[rows, col] = mu + 4 * sigma
    corruption_records.extend({"row_index": r, "column": col} for r in rows)
corruption_records = pd.DataFrame(corruption_records)

diagnostics = mimic.confidence(corrupted, columns=["age", "hours-per-week"])
merged_marker = diagnostics.merge(
    corruption_records.assign(was_corrupted=True),
    on=["row_index", "column"],
    how="left",
)["was_corrupted"]
diagnostics["was_corrupted"] = (merged_marker == True).to_numpy()

ranked = diagnostics.sort_values("discrepancy", ascending=False)
top_k = len(corruption_records)
precision_at_k = ranked.head(top_k)["was_corrupted"].mean()

inconsistency_summary = pd.DataFrame(
    {"metric": ["corrupted_entries", "precision_at_k"], "value": [top_k, precision_at_k]}
)
inconsistency_top = ranked[["row_index", "column", "observed", "prediction", "discrepancy", "was_corrupted"]].head(12)
display(inconsistency_summary.style.format({"value": "{:.3f}"}))
display(inconsistency_top.style.format({"observed": "{:.2f}", "prediction": "{:.2f}", "discrepancy": "{:.2f}"}))

## 3. Supervised learning on income

Income prediction is represented as imputing a missing `income` target column.

In [ ]:
income_task = test_df.copy()
income_truth = test_df["income"].copy()
income_task["income"] = np.nan

income_pred = mimic.impute(income_task, columns=["income"])
y_true = income_truth.astype(str)
y_pred = income_pred["income"].astype(str)
income_confidence = mimic.confidence(income_task, columns=["income"])

income_diagnostics = binary_classification_diagnostics(
    y_true,
    y_pred,
    income_confidence,
    positive_label=">50K",
    negative_label="<=50K",
)
income_examples = pd.DataFrame(
    {"true_income": y_true.head(12).to_numpy(), "predicted_income": y_pred.head(12).to_numpy()}
)

display(income_diagnostics.metrics.style.format({"value": "{:.3f}"}))
display(income_examples)
display(
    income_confidence[["prediction", "confidence", "entropy", "probability_margin", "vote_fraction"]]
    .head(12)
    .style.format(
        {
            "confidence": "{:.3f}",
            "entropy": "{:.3f}",
            "probability_margin": "{:.3f}",
            "vote_fraction": "{:.3f}",
        }
    )
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(income_diagnostics.roc_curve["fpr"], income_diagnostics.roc_curve["tpr"], linewidth=2)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)
axes[0].set_title("ROC curve")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].text(0.62, 0.08, f"AUC = {income_diagnostics.metrics.loc[income_diagnostics.metrics['metric'] == 'roc_auc', 'value'].iat[0]:.3f}")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.25)

axes[1].plot(income_diagnostics.pr_curve["recall"], income_diagnostics.pr_curve["precision"], linewidth=2)
axes[1].axhline(income_diagnostics.positive_rate, linestyle="--", color="black", alpha=0.6)
axes[1].set_title("Precision-recall curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].text(0.58, 0.08, f"AP = {income_diagnostics.metrics.loc[income_diagnostics.metrics['metric'] == 'average_precision', 'value'].iat[0]:.3f}")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.25)
fig.tight_layout()

display(income_diagnostics.confusion_counts.style.set_caption("Confusion table: absolute counts"))
display(
    income_diagnostics.confusion_relative.style
    .format("{:.1%}")
    .set_caption("Confusion table: relative counts")
)


## 4. Synthetic generation

Generate synthetic rows, inspect traceability, and compare original and generated distributions in a shared 2D projection.

In [ ]:
synthetic, trace = mimic.sample(len(test_df), return_trace=True)

display(test_df.head(12).style.set_caption("Held-out test rows"))
display(synthetic.head(12).style.set_caption("Displacement-generated rows"))
display(trace.head(12).style.set_caption("Generation trace"))


In [ ]:
combined = pd.concat(
    [test_df.assign(source="heldout"), synthetic.assign(source="synthetic")],
    ignore_index=True,
)

# Use MIMIC's fitted embedding for both original and generated rows, then project jointly with classical MDS.
H_combined = mimic.transform(combined.drop(columns=["source"]))
plot_df = classical_mds_2d(H_combined, random_state=RANDOM_STATE)
plot_df["source"] = combined["source"].to_numpy()

fig, ax = plt.subplots(figsize=(7, 5))
for source, marker, alpha in [("heldout", "o", 0.3), ("synthetic", "o", 0.3)]:
    part = plot_df[plot_df["source"] == source]
    ax.scatter(part["mds1"], part["mds2"], label=source, marker=marker, alpha=alpha, s=24)
ax.set_title("Held-out Adult rows vs displacement-generated rows in MIMIC embedding MDS")
ax.set_xlabel("MDS 1")
ax.set_ylabel("MDS 2")
ax.legend()
fig.tight_layout()

In [ ]:
summary = pd.concat(
    [test_df[regression_columns].describe().T.add_prefix("heldout_"), synthetic[regression_columns].describe().T.add_prefix("synthetic_")],
    axis=1,
)
comparison_summary = summary[["heldout_mean", "synthetic_mean", "heldout_std", "synthetic_std"]]
display(comparison_summary.style.format("{:.2f}"))

In [ ]:
pair_grid = pairwise_feature_plot(
    test_df,
    synthetic,
    features=regression_columns,
    original_label="heldout",
    generated_label="synthetic",
    max_rows_per_source=300,
    random_state=RANDOM_STATE,
)
pair_grid.fig.suptitle("Pairwise numeric feature statistics: held-out vs displacement-generated", y=1.02)
